# Поведенческие тесты (CheckList)

Три типа тестов из методологии CheckList (Ribeiro et al., ACL 2020):
- **INV** — перефраз (см. `05_robustness_colab.ipynb`): смысл сохранён → ответ не должен меняться.
- **DIR** — отрицание: главный симптом отрицается → уверенность в исходном классе должна падать.
- **MFT** — минимальная функциональность: однозначные жалобы с известной группой.

Здесь считаем DIR и MFT для baseline и трансформера. **Runtime → GPU.**

In [ ]:
%cd /content
!rm -rf anamnesis && git clone -q https://github.com/vinograd131/anamnesis.git
%cd anamnesis
!pip install -q transformers accelerate peft
!pip uninstall -q -y torchao

In [ ]:
import getpass, os
os.environ["MISTRAL_API_KEY"] = getpass.getpass("MISTRAL_API_KEY: ")

## 1. Генерируем отрицательную версию test (`data/test_neg_v1.jsonl`)

In [ ]:
from src.negate import main as build_negation
build_negation()

## 2. Baseline (tf-idf + LogReg): MFT и отрицание

In [ ]:
from src.behavioral import baseline_predictor, run_mft, run_negation
bl_proba, bl_classes = baseline_predictor()
run_mft(bl_proba, bl_classes, "baseline")
run_negation(bl_proba, bl_classes, "baseline")

## 3. Трансформер (RuBioRoBERTa файнтюн): MFT и отрицание

In [ ]:
import torch
from peft import PeftModel
from transformers import AutoModelForSequenceClassification, AutoTokenizer
from src.transformer_ft import MODEL_ID, ID2LABEL, LABEL2ID, _predict_logits, _softmax
from src.mapping import GROUPS

tok = AutoTokenizer.from_pretrained(MODEL_ID)
base = AutoModelForSequenceClassification.from_pretrained(
    MODEL_ID, num_labels=len(GROUPS), id2label=ID2LABEL, label2id=LABEL2ID
)
model = PeftModel.from_pretrained(base, "models/rubioroberta_ft")
device = "cuda" if torch.cuda.is_available() else "cpu"

def tf_proba(texts):
    return _softmax(_predict_logits(model, tok, texts, device, max_length=256))

run_mft(tf_proba, list(GROUPS), "rubioroberta_ft")
run_negation(tf_proba, list(GROUPS), "rubioroberta_ft")

## 4. Графики

In [ ]:
from src.plot_behavioral import main as plot_behavioral
plot_behavioral()

## 5. Сохранить в git

In [ ]:
import getpass
token = getpass.getpass("GitHub token: ")
!git config user.email "karinkakarik@mail.ru"
!git config user.name "Git_Karina"
!git add data/test_neg_v1.jsonl reports/
!git commit -q -m "behavioral tests: negation (DIR) and MFT"
!git pull -q --rebase https://github.com/vinograd131/anamnesis.git main
!git push -q https://{token}@github.com/vinograd131/anamnesis.git HEAD:main